# Kaggle — House Prices : Advanced Regression Techniques

In [1]:
# Recharge automatiquement src/transformers.py quand il est modifié
%load_ext autoreload
%autoreload 2

# Rend le dossier src/ (un niveau au-dessus de notebooks/) importable
import sys
sys.path.append("..")
from src.transformers import GroupMedianImputer, ModeImputer

import pandas as pd
import seaborn as sns
import numpy as np

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
test_ids = test["Id"]  # Id du test, nécessaires pour la soumission

train.shape, test.shape

((1460, 81), (1459, 80))

## 1. Préparation des données

In [2]:
train = train.reset_index(drop=True)

# Cible passée en log (la métrique Kaggle est un RMSE sur le log du prix)
y = np.log1p(train["SalePrice"])

# Features uniquement : on retire l'identifiant et la cible
X_train = train.drop(columns=["Id", "SalePrice"])
X_test  = test.drop(columns=["Id"])

# Concaténation train + test ; n_train sert à re-séparer plus tard
n_train = len(X_train)
all_X = pd.concat([X_train, X_test], axis=0, ignore_index=True)

In [3]:
all_X.shape  # attendu : (2919, 79)

(2919, 79)

## 2. Valeurs manquantes

In [4]:
# Colonnes qui contiennent au moins un NaN
cols_na = all_X.columns[all_X.isnull().any()]
df_null = all_X[cols_na]

# Nombre et pourcentage de NaN, triés par ordre décroissant
na = df_null.isnull().sum().sort_values(ascending=False)
na_pct = (na / len(all_X) * 100).round(1)
pd.DataFrame({"nb_NaN": na, "pct": na_pct})

,nb_NaN,pct
PoolQC,2909,99.7
MiscFeature,2814,96.4
Alley,2721,93.2
Fence,2348,80.4
MasVnrType,1766,60.5
FireplaceQu,1420,48.6
LotFrontage,486,16.6
GarageQual,159,5.4
GarageYrBlt,159,5.4
GarageCond,159,5.4


### 2.1 `PoolQC` — qualité de la piscine (99,7 % de NaN)

D'après `data_description.txt`, `NA` = pas de piscine. On vérifie que les NaN correspondent aux maisons avec `PoolArea == 0`.

In [5]:
# Seulement 10 piscines ont une qualité renseignée
all_X['PoolQC'].value_counts()

PoolQC
Ex    4
Gd    4
Fa    2
Name: count, dtype: int64

**Test de cohérence `PoolQC` × `PoolArea`**

In [6]:
# Maisons sans piscine
(all_X['PoolArea'] == 0).sum()

np.int64(2906)

In [7]:
# Pas de piscine ET pas de qualité -> cohérent
((all_X['PoolArea'] == 0) & (all_X['PoolQC'].isnull())).sum()

np.int64(2906)

In [8]:
# Piscine présente MAIS qualité manquante -> vraies valeurs manquantes
((all_X['PoolArea'] > 0) & (all_X['PoolQC'].isnull())).sum()

np.int64(3)

In [9]:
# Pas de piscine MAIS qualité renseignée -> incohérence
((all_X['PoolArea'] == 0) & (all_X['PoolQC'].notnull())).sum()

np.int64(0)

In [10]:
# Piscine présente ET qualité renseignée -> cohérent
((all_X['PoolArea'] > 0) & (all_X['PoolQC'].notnull())).sum()

np.int64(10)

**Résultat**

| | `PoolQC` vide | `PoolQC` renseigné |
|---|---|---|
| `PoolArea == 0` | 2 906 | 0 |
| `PoolArea > 0` | 3 | 10 |

- 2 906 NaN = pas de piscine.
- 3 piscines sans qualité = vraies valeurs manquantes.
- Aucune incohérence dans l'autre sens.

In [11]:
# Les 3 piscines SANS qualité : train ou test ?
mask = (all_X["PoolArea"] > 0) & (all_X["PoolQC"].isnull())
idx = all_X[mask].index

print(idx)
print(["train" if i < n_train else "test" for i in idx])

Index([2420, 2503, 2599], dtype='int64')
['test', 'test', 'test']


In [12]:
# Les 10 piscines AVEC qualité : train ou test ?
mask = (all_X['PoolArea'] > 0) & (all_X['PoolQC'].notnull())
idx = all_X[mask].index

print(idx)
print(["train" if i < n_train else "test" for i in idx])

Index([197, 810, 1170, 1182, 1298, 1386, 1423, 1974, 2573, 2710], dtype='int64')
['train', 'train', 'train', 'train', 'train', 'train', 'train', 'test', 'test', 'test']


**Conclusion**

- Les 3 piscines sans qualité sont toutes dans le test.
- Seulement 7 piscines notées dans le train sur 1 460 maisons : trop peu pour que le modèle apprenne l'effet de la qualité.
- `PoolArea` contient déjà l'information sur la présence de piscine.

**Décision : suppression de `PoolQC`.**

In [13]:
# Suppression de PoolQC (trop peu d'exemples notés dans le train)
all_X = all_X.drop(columns=["PoolQC"])
all_X.shape  # attendu : (2919, 78)

(2919, 78)

### 2.2 `MiscFeature` — équipement divers (96,4 % de NaN)

D'après `data_description.txt`, `NA` = aucun équipement, et `MiscVal` = valeur en $ de cet équipement. On vérifie que les NaN correspondent aux maisons avec `MiscVal == 0`.

In [14]:
# Répartition des équipements renseignés
all_X['MiscFeature'].value_counts()

MiscFeature
Shed    95
Gar2     5
Othr     4
TenC     1
Name: count, dtype: int64

In [15]:
# Nombre total d'équipements renseignés
all_X['MiscFeature'].value_counts().sum()

np.int64(105)

**Test de cohérence `MiscFeature` × `MiscVal`**

In [16]:
# Maisons sans valeur d'équipement
(all_X['MiscVal'] == 0).sum()

np.int64(2816)

In [17]:
# Pas de valeur ET pas d'équipement -> cohérent
((all_X['MiscVal'] == 0) & (all_X['MiscFeature'].isnull())).sum()

np.int64(2813)

In [18]:
# Valeur renseignée MAIS équipement manquant -> incohérence
((all_X['MiscVal'] > 0) & (all_X['MiscFeature'].isnull())).sum()

np.int64(1)

In [19]:
# Équipement présent MAIS valeur à 0
((all_X['MiscVal'] == 0) & (all_X['MiscFeature'].notnull())).sum()

np.int64(3)

In [20]:
# Équipement présent ET valeur renseignée -> cohérent
((all_X['MiscVal'] > 0) & (all_X['MiscFeature'].notnull())).sum()

np.int64(102)

**Résultat**

| | `MiscFeature` vide | `MiscFeature` renseigné |
|---|---|---|
| `MiscVal == 0` | 2 813 | 3 |
| `MiscVal > 0` | 1 | 102 |

- 2 813 NaN = aucun équipement.
- 1 maison avec une valeur mais sans équipement, 3 équipements avec une valeur à 0.

In [21]:
# Répartition des équipements par catégorie, entre train et test
condition = (all_X['MiscVal'] > 0) & (all_X['MiscFeature'].notnull())
sub = all_X.loc[condition, ['MiscFeature']].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['MiscFeature'], sub['origine'], margins=True)

origine,test,train,All
MiscFeature,,,
Gar2,3,2,5
Othr,2,1,3
Shed,45,48,93
TenC,0,1,1
All,50,52,102


**Conclusion**

- Répartition train / test d'environ 50/50 (52 / 50).
- Dans le train, seule `Shed` a assez d'exemples (48) ; `Gar2`, `Othr` et `TenC` en ont 1 ou 2 : le modèle ne peut rien en apprendre.
- `MiscVal` porte déjà l'information utile (présence d'un équipement et sa valeur) ; la catégorie apporte très peu en plus.
- Garder `MiscFeature` imposerait un encodage pour un gain quasi nul.

**Décision : suppression de `MiscFeature`, on conserve `MiscVal`.**

In [22]:
# Suppression de MiscFeature (MiscVal conserve l'information utile)
all_X = all_X.drop(columns=["MiscFeature"])
all_X.shape  # attendu : (2919, 77)

(2919, 77)

### 2.3 `Alley` — type d'accès par allée (93,2 % de NaN)

D'après `data_description.txt`, `NA` = pas d'accès par une allée. Aucune autre colonne ne décrit l'allée : pas de test de cohérence possible, le NaN se lit directement comme « absent ».

In [23]:
# Répartition des types d'allée renseignés
all_X['Alley'].value_counts()

Alley
Grvl    120
Pave     78
Name: count, dtype: int64

In [24]:
# Nombre total d'allées renseignées
all_X['Alley'].value_counts().sum()

np.int64(198)

In [25]:
# Répartition des types d'allée entre train et test
sub = all_X[["Alley"]].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['Alley'], sub['origine'], margins=True)

origine,test,train,All
Alley,,,
Grvl,70,50,120
Pave,37,41,78
All,107,91,198


**Conclusion**

- Effectifs suffisants dans le train : `Grvl` 50, `Pave` 41.
- Pas d'ordre entre gravier et pavé : encodage one-hot (pas ordinal).
- Aucune autre colonne ne porte cette information.

**Décision : NaN remplacés par `"None"`.**
Le one-hot sera fait à la fin, en une seule fois sur toutes les colonnes catégorielles ; la colonne `Alley_None` pourra alors être supprimée (redondante avec `Alley_Grvl = Alley_Pave = 0`).

In [26]:
# NaN = pas d'allée
all_X["Alley"] = all_X["Alley"].fillna("None")

In [27]:
# Vérification : plus aucun NaN dans Alley
all_X['Alley'].value_counts()

Alley
None    2721
Grvl     120
Pave      78
Name: count, dtype: int64

### 2.4 `Fence` — qualité de la clôture (80,4 % de NaN)

D'après `data_description.txt`, `NA` = pas de clôture. Valeurs : `GdPrv` (bonne intimité), `MnPrv` (intimité minimale), `GdWo` (bon bois), `MnWw` (bois/grillage minimal). Aucune autre colonne ne décrit la clôture : le NaN se lit directement comme « absent ».

In [28]:
# Répartition des types de clôture renseignés
all_X['Fence'].value_counts()

Fence
MnPrv    329
GdPrv    118
GdWo     112
MnWw      12
Name: count, dtype: int64

In [29]:
# Nombre total de clôtures renseignées
all_X['Fence'].value_counts().sum()

np.int64(571)

In [30]:
# Répartition des types de clôture entre train et test
sub = all_X[["Fence"]].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['Fence'], sub['origine'], margins=True)

origine,test,train,All
Fence,,,
GdPrv,59,59,118
GdWo,58,54,112
MnPrv,172,157,329
MnWw,1,11,12
All,290,281,571


**Conclusion**

- Répartition train / test d'environ 50/50 (281 / 290).
- Effectifs suffisants dans le train sauf `MnWw` (11).
- Deux familles de clôture (intimité / bois), chacune avec un niveau minimum et un niveau bon.

**Décision : NaN remplacés par `"None"`.**
Encodage à l'étape finale. Piste à comparer en CV avec un one-hot classique : deux colonnes ordinales `Fence_Privacy` (None 0, MnPrv 1, GdPrv 2) et `Fence_Wood` (None 0, MnWw 1, GdWo 2), possibles car chaque maison a au plus une clôture.

In [31]:
# NaN = pas de clôture, puis vérification
all_X["Fence"] = all_X["Fence"].fillna("None")
all_X['Fence'].value_counts()

Fence
None     2348
MnPrv     329
GdPrv     118
GdWo      112
MnWw       12
Name: count, dtype: int64

### 2.5 `MasVnrType` — type de placage maçonnerie (60,5 % de NaN)

D'après `data_description.txt`, `MasVnrType` n'a pas de valeur `NA` mais une valeur `None` (pas de placage) : pandas convertit automatiquement le texte `"None"` en NaN à la lecture. `MasVnrArea` (surface du placage) a aussi 23 NaN. On croise les deux colonnes.

In [32]:
# Répartition des types de placage renseignés
all_X['MasVnrType'].value_counts()

MasVnrType
BrkFace    879
Stone      249
BrkCmn      25
Name: count, dtype: int64

In [33]:
# Croise 'colonne catégorielle vide / renseignée' avec 'colonne numérique NaN / 0 / > 0'
# (réutilisable pour les groupes Garage*, Bsmt*, Fireplace*...)
def coherence(df, col_cat, col_num):
    etat_cat = df[col_cat].isnull().map({True: "vide", False: "renseigné"}).rename(col_cat)
    etat_num = pd.Series(
        np.select([df[col_num].isnull(), df[col_num] == 0], ["NaN", "0"], default="> 0"),
        index=df.index, name=col_num
    )
    return pd.crosstab(etat_cat, etat_num, margins=True)

In [34]:
# Test de cohérence MasVnrType × MasVnrArea
coherence(all_X, "MasVnrType", "MasVnrArea")

MasVnrArea,0,> 0,NaN,All
MasVnrType,,,,
renseigné,3,1150,0,1153
vide,1735,8,23,1766
All,1738,1158,23,2919


In [35]:
# Répartition des types de placage entre train et test
sub = all_X[["MasVnrType"]].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['MasVnrType'], sub['origine'], margins=True)

origine,test,train,All
MasVnrType,,,
BrkCmn,10,15,25
BrkFace,434,445,879
Stone,121,128,249
All,565,588,1153


**Conclusion**

- 1 735 maisons sans type et sans surface : pas de placage.
- 23 maisons sans type ni surface : vraies valeurs manquantes sur les deux colonnes ; la majorité des maisons n'ayant pas de placage, on les considère sans placage.
- 8 maisons avec une surface mais sans type : la surface prouve qu'il y a un placage. On leur attribuera le type le plus fréquent **parmi les maisons qui ont un placage** ; ce mode est une statistique apprise, il sera donc calculé dans le **Pipeline** (sur la partie entraînement de chaque fold) pour éviter toute fuite.
- 3 maisons avec un type mais une surface à 0 : laissées telles quelles (pas de NaN, impossible de savoir quelle colonne est fausse).
- Pas d'ordre entre les types ; `BrkCmn` a 15 exemples dans le train, suffisant pour le garder.

**Décision** : `MasVnrType` → `"None"` pour les maisons sans placage, `MasVnrArea` → 0. Les 8 maisons restent NaN et seront imputées dans le Pipeline. One-hot à l'étape finale.

In [36]:
# Pas de type ET (surface à 0 ou vide) -> pas de placage
masque = ( ((all_X["MasVnrArea"] == 0) |(all_X["MasVnrArea"].isna()) )  & (all_X["MasVnrType"].isna() )) 

all_X.loc[masque, "MasVnrType"] = "None"
# Les 8 maisons avec une surface > 0 restent NaN : imputées dans le Pipeline

In [37]:
# Surface manquante = pas de placage
all_X["MasVnrArea"] = all_X["MasVnrArea"].fillna(0)

In [38]:
# Vérification : None uniquement avec une surface à 0, 8 NaN restants (surface > 0)
etat_area = np.where(all_X["MasVnrArea"] > 0, "> 0", "0")
pd.crosstab(all_X["MasVnrType"], etat_area, margins=True, colnames=["MasVnrArea"], dropna=False)

MasVnrArea,0,> 0,All
MasVnrType,,,
BrkCmn,0,25,25
BrkFace,2,877,879
None,1758,0,1758
Stone,1,248,249
NaN,0,8,8
All,1761,1158,2919


### 2.6 `FireplaceQu` — qualité de la cheminée (48,6 % de NaN)

D'après `data_description.txt`, `NA` = pas de cheminée. Échelle de qualité : `Po` < `Fa` < `TA` < `Gd` < `Ex`. On vérifie la cohérence avec `Fireplaces` (nombre de cheminées).

In [39]:
# Répartition des niveaux de qualité renseignés
all_X['FireplaceQu'].value_counts()

FireplaceQu
Gd    744
TA    592
Fa     74
Po     46
Ex     43
Name: count, dtype: int64

In [40]:
# Nombre total de cheminées avec une qualité renseignée
all_X['FireplaceQu'].value_counts().sum()

np.int64(1499)

In [41]:
# Test de cohérence FireplaceQu × Fireplaces
coherence(all_X, "FireplaceQu", "Fireplaces")

Fireplaces,0,> 0,All
FireplaceQu,,,
renseigné,0,1499,1499
vide,1420,0,1420
All,1420,1499,2919


**Conclusion**

- Cohérence parfaite : les 1 420 NaN correspondent exactement aux maisons sans cheminée, les 1 499 qualités renseignées aux maisons avec au moins une cheminée.
- Échelle de qualité ordonnée : encodage **ordinal** avec le dictionnaire commun des qualités (`None` 0, `Po` 1, `Fa` 2, `TA` 3, `Gd` 4, `Ex` 5). Les niveaux rares (`Po`, `Ex`) ne posent pas de problème en ordinal.

**Décision : NaN remplacés par `"None"`.** Encodage ordinal à l'étape finale.

In [42]:
# NaN = pas de cheminée, puis vérification
all_X["FireplaceQu"] = all_X["FireplaceQu"].fillna("None")
all_X['FireplaceQu'].value_counts()

FireplaceQu
None    1420
Gd       744
TA       592
Fa        74
Po        46
Ex        43
Name: count, dtype: int64

### 2.7 `LotFrontage` — longueur de façade sur rue (16,6 % de NaN)

Chaque terrain a une façade : les NaN sont de **vraies valeurs manquantes**, pas des « absents ». Il faut donc les imputer.

In [43]:
# Distribution sur le train
all_X.iloc[:n_train]['LotFrontage'].describe()

count    1201.000000
mean       70.049958
std        24.284752
min        21.000000
25%        59.000000
50%        69.000000
75%        80.000000
max       313.000000
Name: LotFrontage, dtype: float64

In [44]:
# Médiane globale sur le train
all_X.iloc[:n_train]['LotFrontage'].median()

np.float64(69.0)

In [45]:
# Médiane et effectif par quartier sur le train
all_X.iloc[:n_train].groupby("Neighborhood")["LotFrontage"].agg(["median", "count"])

,median,count
Neighborhood,,
Blmngtn,43.0,14
Blueste,24.0,2
BrDale,21.0,16
BrkSide,52.0,51
ClearCr,80.0,13
CollgCr,70.0,126
Crawfor,74.0,41
Edwards,65.5,92
Gilbert,65.0,49


**Conclusion de l'analyse**

- Moyenne (70,0) et médiane (69) proches sur le train ; la médiane reste préférée, plus robuste aux valeurs extrêmes (max 313).
- La façade varie fortement selon le quartier : médianes de 21 (`BrDale`, `MeadowV`) à 91 (`NoRidge`). Une médiane globale serait très imprécise.
- Certains quartiers ont très peu de terrains dans le train (`Blueste` : 2) : une médiane globale de secours est nécessaire pour un quartier absent des données d'apprentissage.

**Décision : imputation par la médiane du quartier, médiane globale en secours.**
Cette médiane est une statistique apprise : elle est calculée dans un transformer scikit-learn (`GroupMedianImputer`, dans `src/transformers.py`) intégré au **Pipeline**, pour être réapprise sur la partie entraînement de chaque fold de la CV, sans fuite. Les NaN restent donc dans `all_X`.

In [46]:
# Test du transformer : fit sur le train uniquement
imp = GroupMedianImputer()
imp.fit(all_X.iloc[:n_train])

print(imp.medians_)        # doit correspondre au groupby ci-dessus
print(imp.global_median_)  # doit valoir 69.0

Neighborhood
Blmngtn    43.0
Blueste    24.0
BrDale     21.0
BrkSide    52.0
ClearCr    80.0
CollgCr    70.0
Crawfor    74.0
Edwards    65.5
Gilbert    65.0
IDOTRR     60.0
MeadowV    21.0
Mitchel    73.0
NAmes      73.0
NPkVill    24.0
NWAmes     80.0
NoRidge    91.0
NridgHt    88.5
OldTown    60.0
SWISU      60.0
Sawyer     71.0
SawyerW    66.5
Somerst    73.5
StoneBr    61.5
Timber     85.0
Veenker    68.0
Name: LotFrontage, dtype: float64
69.0


In [47]:
# Test du transformer : transform sur tout all_X
res = imp.transform(all_X)
print(res["LotFrontage"].isna().sum())     # attendu : 0 (tout est rempli)
print(all_X["LotFrontage"].isna().sum())   # attendu : 486 (all_X d'origine intact)

0
486


### 2.8 `MasVnrType` — imputation des 8 maisons avec surface mais sans type

Après le nettoyage de la section 2.5, les seuls NaN restants de `MasVnrType` sont les 8 maisons avec `MasVnrArea > 0`. Leur type est imputé par le type le plus fréquent **parmi les maisons qui ont un placage**.

Le mode de toute la colonne ne convient pas : `"None"` est désormais une catégorie (1 758 maisons), elle serait la plus fréquente. Le transformer `ModeImputer` (dans `src/transformers.py`) exclut donc cette valeur avant de calculer le mode, et sera intégré au Pipeline.

In [48]:
# Test du transformer : fit sur le train uniquement
mode_imp = ModeImputer()
mode_imp.fit(all_X.iloc[:n_train])
print(mode_imp.mode_)    # attendu : BrkFace

BrkFace


In [49]:
# Test du transformer : transform sur tout all_X
res = mode_imp.transform(all_X)
print(res["MasVnrType"].isna().sum())       # attendu : 0
print(res["MasVnrType"].value_counts())     # BrkFace : 879 -> 887
print(all_X["MasVnrType"].isna().sum())     # attendu : 8 (all_X d'origine intact)

0
MasVnrType
None       1758
BrkFace     887
Stone       249
BrkCmn       25
Name: count, dtype: int64
8


**Conclusion**

- Le mode appris sur le train (hors `"None"`) est `BrkFace` ; les 8 maisons passent en `BrkFace` (879 → 887).
- Plus aucun NaN dans `MasVnrType` après le transformer, et `all_X` d'origine reste intact.
- Hypothèse du transformer : le nettoyage de la section 2.5 est fait avant lui (seules les maisons à imputer sont encore NaN).

### 2.9 Groupe `Garage*` (≈ 5,4 % de NaN)

7 colonnes liées : `GarageType`, `GarageFinish`, `GarageQual`, `GarageCond`, `GarageYrBlt` (NA = pas de garage), `GarageArea`, `GarageCars`. On prend `GarageArea` comme colonne de référence (0 = pas de garage) et on croise chaque colonne avec elle.

In [50]:
# Distribution de la surface de garage
all_X['GarageArea'].describe()

count    2918.000000
mean      472.874572
std       215.394815
min         0.000000
25%       320.000000
50%       480.000000
75%       576.000000
max      1488.000000
Name: GarageArea, dtype: float64

In [51]:
# Maisons avec une surface de garage à 0
(all_X['GarageArea'] == 0).value_counts()

GarageArea
False    2762
True      157
Name: count, dtype: int64

In [52]:
# Maison(s) sans surface renseignée
all_X[all_X["GarageArea"].isna()].index.tolist()

[2576]

In [53]:
# Maison(s) sans nombre de voitures renseigné
all_X[all_X["GarageCars"].isna()].index.tolist()

[2576]

In [54]:
# Test de cohérence GarageType × GarageArea
coherence(all_X, "GarageType", "GarageArea")

GarageArea,0,> 0,NaN,All
GarageType,,,,
renseigné,0,2761,1,2762
vide,157,0,0,157
All,157,2761,1,2919


In [55]:
# Test de cohérence GarageFinish × GarageArea
coherence(all_X, "GarageFinish", "GarageArea")

GarageArea,0,> 0,NaN,All
GarageFinish,,,,
renseigné,0,2760,0,2760
vide,157,1,1,159
All,157,2761,1,2919


In [56]:
# Garages existants (surface > 0 ou inconnue) sans finition
masque = (
    ((all_X["GarageArea"] > 0) | (all_X["GarageArea"].isna()))
    & (all_X["GarageFinish"].isna())
)
all_X.loc[masque, "GarageFinish"].index.tolist()

[2126, 2576]

In [57]:
# Test de cohérence GarageQual × GarageArea
coherence(all_X, "GarageQual", "GarageArea")

GarageArea,0,> 0,NaN,All
GarageQual,,,,
renseigné,0,2760,0,2760
vide,157,1,1,159
All,157,2761,1,2919


In [58]:
# Garages existants (surface > 0 ou inconnue) sans qualité
masque = (
    ((all_X["GarageArea"] > 0) | (all_X["GarageArea"].isna()))
    & (all_X["GarageQual"].isna())
)
all_X.loc[masque, "GarageQual"].index.tolist()

[2126, 2576]

In [59]:
# Test de cohérence GarageCond × GarageArea
coherence(all_X, "GarageCond", "GarageArea")

GarageArea,0,> 0,NaN,All
GarageCond,,,,
renseigné,0,2760,0,2760
vide,157,1,1,159
All,157,2761,1,2919


In [60]:
# Garages existants (surface > 0 ou inconnue) sans état
masque = (
    ((all_X["GarageArea"] > 0) | (all_X["GarageArea"].isna()))
    & (all_X["GarageCond"].isna())
)
all_X.loc[masque, "GarageCond"].index.tolist()

[2126, 2576]

In [61]:
# Test de cohérence GarageYrBlt × GarageArea
coherence(all_X, "GarageYrBlt", "GarageArea")

GarageArea,0,> 0,NaN,All
GarageYrBlt,,,,
renseigné,0,2760,0,2760
vide,157,1,1,159
All,157,2761,1,2919


In [62]:
# Garages existants (surface > 0 ou inconnue) sans année
masque = (
    ((all_X["GarageArea"] > 0) | (all_X["GarageArea"].isna()))
    & (all_X["GarageYrBlt"].isna())
)
all_X.loc[masque, "GarageYrBlt"].index.tolist()

[2126, 2576]

**Constat**

- 157 maisons sans garage, parfaitement cohérentes (surface 0, toutes les colonnes vides).
- `GarageType` a 157 NaN contre 159 pour les autres : deux maisons ont un type mais pas le reste.
  - **2126** : surface 360, 1 voiture, type `Detchd`, mais pas de finition, qualité, état ni année.
  - **2576** : type `Detchd`, tout le reste vide (y compris surface et voitures).

In [63]:
# Type de garage de la maison sans surface (2576)
all_X.loc[all_X["GarageArea"].isna(), "GarageType"] 

2576    Detchd
Name: GarageType, dtype: str

In [64]:
# Répartition des types de garage
all_X["GarageType"].value_counts()

GarageType
Attchd     1723
Detchd      779
BuiltIn     186
Basment      36
2Types       23
CarPort      15
Name: count, dtype: int64

In [65]:
# Les deux maisons incohérentes, toutes colonnes Garage*
cols_garage = [c for c in all_X.columns if c.startswith("Garage")]
all_X.loc[[2126, 2576], cols_garage]

,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond
2126,Detchd,NaN,NaN,1.0,360.0,NaN,NaN
2576,Detchd,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
# 2126 : train ou test ?
idx = 2126  

if idx < len(X_train):
    print(f"L'index {idx} est dans X_train")
else:
    print(f"L'index {idx} est dans X_test ")

L'index 2126 est dans X_test 


In [67]:
# 2576 : train ou test ?
idx = 2576  

if idx < len(X_train):
    print(f"L'index {idx} est dans X_train")
else:
    print(f"L'index {idx} est dans X_test ")

L'index 2576 est dans X_test 


In [68]:
# Contexte des deux maisons
cols = ["YearBuilt", "OverallQual", "GrLivArea"]
all_X.loc[[2126, 2576], cols]

,YearBuilt,OverallQual,GrLivArea
2126,1910,6,1984
2576,1923,5,1828


In [69]:
# Distribution des années de construction
all_X["YearBuilt"].describe()

count    2919.000000
mean     1971.312778
std        30.291442
min      1872.000000
25%      1953.500000
50%      1973.000000
75%      2001.000000
max      2010.000000
Name: YearBuilt, dtype: float64

**Décision pour 2126 et 2576**

- Les deux sont dans le **test** : la décision ne change pas l'apprentissage, seulement 2 prédictions.
- Profils très proches (garage `Detchd`, maisons de 1910 et 1923, qualité moyenne) ; les garages détachés sont courants à cette époque.
- **2126** : la surface prouve que le garage existe.
- **2576** : hypothèse A retenue — elle a un garage détaché dont les détails sont perdus (on garde la seule information connue, le type).

Les valeurs manquantes de ces deux maisons sont des statistiques apprises : elles seront imputées dans le **Pipeline**. Reste à choisir mode ou médiane pour les colonnes ordinales.

In [70]:
# Répartition de GarageQual sur le train
all_X.iloc[:n_train]["GarageQual"].value_counts()

GarageQual
TA    1311
Fa      48
Gd      14
Ex       3
Po       3
Name: count, dtype: int64

In [71]:
# Répartition de GarageCond sur le train
all_X.iloc[:n_train]["GarageCond"].value_counts()

GarageCond
TA    1326
Fa      35
Gd       9
Po       7
Ex       2
Name: count, dtype: int64

In [72]:
# Répartition de GarageFinish sur le train
all_X.iloc[:n_train]["GarageFinish"].value_counts()

GarageFinish
Unf    605
RFn    422
Fin    352
Name: count, dtype: int64

In [73]:
# Répartition de GarageQual pour les garages détachés
all_X.loc[all_X["GarageType"] == "Detchd" , "GarageQual"].value_counts()

GarageQual
TA    667
Fa     97
Po      5
Gd      5
Ex      3
Name: count, dtype: int64

In [74]:
# Répartition de GarageCond pour les garages détachés
all_X.loc[all_X["GarageType"] == "Detchd" , "GarageCond"].value_counts()

GarageCond
TA    692
Fa     67
Po     12
Ex      3
Gd      3
Name: count, dtype: int64

In [75]:
# Répartition de GarageFinish pour les garages détachés
all_X.loc[all_X["GarageType"] == "Detchd" , "GarageFinish"].value_counts()

GarageFinish
Unf    719
RFn     34
Fin     24
Name: count, dtype: int64

In [76]:
# Répartition de GarageFinish pour les garages détachés du train
train_X = all_X.iloc[:n_train]
train_X.loc[train_X["GarageType"] == "Detchd", "GarageFinish"].value_counts()

GarageFinish
Unf    354
RFn     19
Fin     14
Name: count, dtype: int64

**Mode ou médiane ?**

- `GarageQual` et `GarageCond` : `TA` représente ≈ 95 % des garages du train → mode = médiane.
- `GarageFinish` : sur tous les garages du train, `Unf` est le plus fréquent (44 %) mais pas majoritaire, la médiane globale serait `RFn`. Pour les garages **détachés**, `Unf` représente 91 %.
- Les deux maisons à imputer étant `Detchd`, le **mode global (`Unf`) est plus représentatif** que la médiane globale : on utilise `ModeImputer` pour les trois colonnes.

In [77]:
# Pas de garage (surface à 0) -> "None"
# Le masque n'inclut pas les surfaces vides : 2576 a un garage (hypothèse A)
cols_garage_cat = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
masque = (all_X["GarageArea"] == 0)
all_X.loc[masque, cols_garage_cat] = "None"

In [78]:
# Vérification : seules 2126 et 2576 restent vides
coherence(all_X, "GarageFinish", "GarageArea")

GarageArea,0,> 0,NaN,All
GarageFinish,,,,
renseigné,157,2760,0,2917
vide,0,1,1,2
All,157,2761,1,2919


In [79]:
# GarageArea de 2576 : médiane par type de garage, apprise sur le train
imp2 = GroupMedianImputer(target_col="GarageArea", group_col="GarageType")
imp2.fit(all_X.iloc[:n_train])
res2 = imp2.transform(all_X)
print(imp2.medians_)    
print(imp2.global_median_)     

GarageType
2Types     825.0
Attchd     500.0
Basment    450.0
BuiltIn    553.0
CarPort    462.0
Detchd     400.0
None         0.0
Name: GarageArea, dtype: float64
480.0


In [80]:
# GarageCars de 2576 : médiane par type de garage, apprise sur le train
# (médianes entières : pas de problème de virgule)
imp3 = GroupMedianImputer(target_col="GarageCars", group_col="GarageType")
imp3.fit(all_X.iloc[:n_train])
res2 = imp3.transform(imp2.transform(all_X))
print(imp3.medians_)    
print(imp3.global_median_)     

GarageType
2Types     3.0
Attchd     2.0
Basment    2.0
BuiltIn    2.0
CarPort    2.0
Detchd     2.0
None       0.0
Name: GarageCars, dtype: float64
2.0


In [81]:
# Résultat pour les deux maisons
cols_garage = [c for c in all_X.columns if c.startswith("Garage")]
res2.loc[[2126, 2576], cols_garage]

,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond
2126,Detchd,NaN,NaN,1.0,360.0,NaN,NaN
2576,Detchd,NaN,NaN,2.0,400.0,NaN,NaN


In [82]:
# GarageFinish, GarageQual, GarageCond : mode du train hors "None"
mode_imp2 = ModeImputer(target_col = "GarageFinish")
mode_imp2.fit(all_X.iloc[:n_train])

mode_imp3 = ModeImputer(target_col = "GarageQual")
mode_imp3.fit(all_X.iloc[:n_train])

mode_imp4 = ModeImputer(target_col = "GarageCond")
mode_imp4.fit(all_X.iloc[:n_train])

res3 = mode_imp2.transform(mode_imp3.transform(mode_imp4.transform(all_X)))


cols_garage = [c for c in all_X.columns if c.startswith("Garage")]
res3.loc[[2126, 2576], cols_garage]

,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond
2126,Detchd,NaN,Unf,1.0,360.0,TA,TA
2576,Detchd,NaN,Unf,NaN,NaN,TA,TA


In [83]:
# Vérification : plus aucune maison vide
coherence(res3, "GarageFinish", "GarageArea")

GarageArea,0,> 0,NaN,All
GarageFinish,,,,
renseigné,157,2761,1,2919
All,157,2761,1,2919


In [84]:
# Vérification : plus aucune maison vide
coherence(res3, "GarageQual", "GarageArea")

GarageArea,0,> 0,NaN,All
GarageQual,,,,
renseigné,157,2761,1,2919
All,157,2761,1,2919


In [85]:
# Vérification : plus aucune maison vide
coherence(res3, "GarageCond", "GarageArea")

GarageArea,0,> 0,NaN,All
GarageCond,,,,
renseigné,157,2761,1,2919
All,157,2761,1,2919


In [86]:
# Enchaînement des 5 transformers, comme dans le Pipeline
res = imp2.transform(all_X)
res = imp3.transform(res)
res = mode_imp2.transform(res)
res = mode_imp3.transform(res)
res = mode_imp4.transform(res)

res[cols_garage].isna().sum()   # attendu : 0 partout sauf GarageYrBlt (159)

GarageType        0
GarageYrBlt     159
GarageFinish      0
GarageCars        0
GarageArea        0
GarageQual        0
GarageCond        0
dtype: int64

**`GarageYrBlt`**

Une année ne peut recevoir ni `"None"` ni `0` (valeur extrême qui fausserait les modèles linéaires). On vérifie d'abord la colonne.

In [87]:
# Distribution des années de construction du garage (max anormal)
all_X["GarageYrBlt"].describe()

count    2760.000000
mean     1978.113406
std        25.574285
min      1895.000000
25%      1960.000000
50%      1979.000000
75%      2002.000000
max      2207.000000
Name: GarageYrBlt, dtype: float64

In [88]:
# Ligne avec l'année maximale
all_X.loc[all_X["GarageYrBlt"] == all_X["GarageYrBlt"].max()]

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,Fence,MiscVal,MoSold,YrSold,SaleType,SaleCondition
2592,20,RL,68.0,8298,Pave,None,IR1,HLS,AllPub,Inside,...,0,0,0,0,None,0,9,2007,New,Partial


In [89]:
# 2592 : train ou test ?
idx = 2592  

if idx < len(X_train):
    print(f"L'index {idx} est dans X_train")
else:
    print(f"L'index {idx} est dans X_test ")

L'index 2592 est dans X_test 


In [90]:
# Années de la maison 2592
all_X.loc[2592, ["YearBuilt", "YearRemodAdd", "GarageYrBlt", "YrSold"]]

YearBuilt       2006.0
YearRemodAdd    2007.0
GarageYrBlt     2207.0
YrSold          2007.0
Name: 2592, dtype: float64

In [93]:
# Correction de la faute de frappe : 2207 -> 2007
all_X.loc[2592, "GarageYrBlt"] = 2007

In [94]:
# Garages construits après la vente
masque = all_X["GarageYrBlt"] > all_X["YrSold"]
col_verif_anne = ["YearBuilt", "YearRemodAdd", "GarageYrBlt", "YrSold"]
all_X.loc[masque,col_verif_anne]

,YearBuilt,YearRemodAdd,GarageYrBlt,YrSold
2549,2008,2009,2008.0,2007


In [96]:
# Corrélation entre année du garage et année de la maison
correlation = all_X["GarageYrBlt"].corr(all_X["YearBuilt"])
print(correlation)

0.8437727896759312


In [97]:
# Écart entre année du garage et année de la maison
diff = all_X["GarageYrBlt"] - all_X["YearBuilt"]
print(diff.describe())

count    2760.000000
mean        5.153261
std        15.825940
min       -20.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       123.000000
dtype: float64


**Conclusion**

- **2592** (test) : `GarageYrBlt = 2207` est une faute de frappe → **2007** (maison construite en 2006, rénovée et vendue en 2007, vente `New` / `Partial`).
- **2549** : vendue en 2007 avant d'être construite en 2008 (vente sur plan) ; années de construction cohérentes entre elles → laissée telle quelle.
- `GarageYrBlt` est très redondante avec `YearBuilt` : corrélation 0,84, et au moins 75 % des garages construits la même année que la maison (médiane de l'écart = 0). Il faudrait en plus inventer 159 valeurs, dont 157 pour des maisons sans garage.

**Décision : suppression de `GarageYrBlt`.**

In [98]:
# Suppression de GarageYrBlt (redondante avec YearBuilt)
all_X = all_X.drop(columns=["GarageYrBlt"])
all_X.shape 

(2919, 76)

### 2.10 Groupe `Bsmt*` — sous-sol